In [1]:
import pandas as pd

## Ingestion

In [2]:
df = pd.read_csv('../data/feature_store_data.csv')
df

,id,feature_name,feature_group,computation_logic,data_source,update_frequency,serving_store,models_using_feature,feature_description
0,0,purchase_count_7d,customer_behavior,COUNT(order_id) WHERE order_status='DELIVERED'...,fct_orders (Silver),Hourly,"DynamoDB, S3","recommendation_model, churn_model",Number of delivered orders placed by a custome...
1,1,purchase_count_14d,customer_behavior,COUNT(order_id) WHERE order_status='DELIVERED'...,fct_orders (Silver),Hourly,"DynamoDB, S3","recommendation_model, churn_model",Number of delivered orders placed by a custome...
2,2,purchase_count_30d,customer_behavior,COUNT(order_id) WHERE order_status='DELIVERED'...,fct_orders (Silver),Hourly,"DynamoDB, S3","recommendation_model, churn_model",Number of delivered orders placed by a custome...
3,3,avg_order_value_7d,customer_spend,AVG(order_total) WHERE order_status='DELIVERED...,fct_orders (Silver),Daily,"DynamoDB, S3","ltv_model, recommendation_model",Average delivered order value in the last 7 da...
4,4,avg_order_value_14d,customer_spend,AVG(order_total) WHERE order_status='DELIVERED...,fct_orders (Silver),Daily,"DynamoDB, S3","ltv_model, recommendation_model",Average delivered order value in the last 14 d...
...,...,...,...,...,...,...,...,...,...
235,235,review_count_14d_v2,product_quality,COUNT(review_id) OVER last 14 days BY product_id,fct_reviews,Daily,DynamoDB,"ranking_model, recommendation_model",Number of product reviews in the last 14 days....
236,236,review_count_30d_v2,product_quality,COUNT(review_id) OVER last 30 days BY product_id,fct_reviews,Daily,DynamoDB,"ranking_model, recommendation_model",Number of product reviews in the last 30 days....
237,237,ad_click_rate_7d_v2,advertising,COUNT(clicks)/COUNT(impressions) OVER last 7 d...,fct_ad_events,Hourly,Redis,"marketing_model, recommendation_model",Ad click-through rate in the last 7 days. This...
238,238,ad_click_rate_14d_v2,advertising,COUNT(clicks)/COUNT(impressions) OVER last 14 ...,fct_ad_events,Hourly,Redis,"marketing_model, recommendation_model",Ad click-through rate in the last 14 days. Thi...


In [3]:
df.columns

Index(['id', 'feature_name', 'feature_group', 'computation_logic',
       'data_source', 'update_frequency', 'serving_store',
       'models_using_feature', 'feature_description'],
      dtype='str')

In [4]:
import minsearch
from minsearch import Index

# Create the MinSearch index with your text and keyword fields
# Define which fields should be searchable
index = minsearch.Index(text_fields = [
    "feature_name",
    "feature_group", 
    "computation_logic",
    "data_source",
    "update_frequency",
    "serving_store",
    "models_using_feature",
    "feature_description"
], keyword_fields = ["id"])

# Convert DataFrame to list of dictionaries
documents = df.to_dict(orient="records")

# Create and fit the index
index.fit(documents)

In [5]:
query = 'what are the names of feature to show click rate'
index.search(query, num_results = 7)

[{'id': 114,
  'feature_name': 'click_rate_7d',
  'feature_group': 'marketing',
  'computation_logic': 'COUNT(clicks)/COUNT(impressions) OVER last 7 days BY campaign_id',
  'data_source': 'fct_marketing_events',
  'update_frequency': 'Hourly',
  'serving_store': 'DynamoDB',
  'models_using_feature': 'marketing_model, recommendation_model',
  'feature_description': 'Campaign click-through rate over the last 7 days. This feature is computed from fct_marketing_events and is commonly used by marketing_model, recommendation_model to capture recent customer, product, or operational behavior.'},
 {'id': 54,
  'feature_name': 'click_rate_7d',
  'feature_group': 'marketing',
  'computation_logic': 'COUNT(clicks)/COUNT(impressions) OVER last 7 days BY campaign_id',
  'data_source': 'fct_marketing_events',
  'update_frequency': 'Hourly',
  'serving_store': 'DynamoDB',
  'models_using_feature': 'marketing_model, recommendation_model',
  'feature_description': 'Campaign click-through rate over the 

## RAG flow

In [6]:
from openai import OpenAI
import os
from dotenv import load_dotenv

# Load .envrc (or .env) file
load_dotenv('../.envrc')
client = OpenAI()

In [7]:
def search(query):
    boost = {}

    results = index.search(
        query=query,
        filter_dict={},
        boost_dict=boost,
        num_results=10
    )

    return results

In [8]:
documents[0]

{'id': 0,
 'feature_name': 'purchase_count_7d',
 'feature_group': 'customer_behavior',
 'computation_logic': "COUNT(order_id) WHERE order_status='DELIVERED' OVER last 7 days BY customer_id",
 'data_source': 'fct_orders (Silver)',
 'update_frequency': 'Hourly',
 'serving_store': 'DynamoDB, S3',
 'models_using_feature': 'recommendation_model, churn_model',
 'feature_description': 'Number of delivered orders placed by a customer in the last 7 days. This feature is computed from fct_orders (Silver) and is commonly used by recommendation_model, churn_model to capture recent customer, product, or operational behavior.'}

In [9]:
prompt_template = """
You are a feature store documentation assistant for an Amazon online shopping platform.
Answer the QUESTION based on the CONTEXT from our feature store documentation.
Use only the facts from the CONTEXT when answering the QUESTION.

QUESTION: {question}

CONTEXT:
{context}
""".strip()

entry_template = """
feature_name: {feature_name}
feature_group: {feature_group}
computation_logic: {computation_logic}
data_source: {data_source}
update_frequency: {update_frequency}
serving_store: {serving_store}
models_using_feature: {models_using_feature}
feature_description : {feature_description}
""".strip()

def build_prompt(query, search_results):
    context = ""
    
    for doc in search_results:
        context = context + entry_template.format(**doc) + "\n\n"

    prompt = prompt_template.format(question=query, context=context).strip()
    return prompt

In [10]:
search_results = search(query)
prompt = build_prompt(query, search_results)

In [11]:
print(prompt)

You are a feature store documentation assistant for an Amazon online shopping platform.
Answer the QUESTION based on the CONTEXT from our feature store documentation.
Use only the facts from the CONTEXT when answering the QUESTION.

QUESTION: what are the names of feature to show click rate

CONTEXT:
feature_name: click_rate_7d
feature_group: marketing
computation_logic: COUNT(clicks)/COUNT(impressions) OVER last 7 days BY campaign_id
data_source: fct_marketing_events
update_frequency: Hourly
serving_store: DynamoDB
models_using_feature: marketing_model, recommendation_model
feature_description : Campaign click-through rate over the last 7 days. This feature is computed from fct_marketing_events and is commonly used by marketing_model, recommendation_model to capture recent customer, product, or operational behavior.

feature_name: click_rate_7d
feature_group: marketing
computation_logic: COUNT(clicks)/COUNT(impressions) OVER last 7 days BY campaign_id
data_source: fct_marketing_events

In [12]:
def llm(prompt, model='gpt-4o-mini'):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

In [13]:
def rag(query, model='gpt-4o-mini'):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    #print(prompt)
    answer = llm(prompt, model=model)
    return answer

In [14]:
query = "What is the difference between the features 'search_to_purchase_conversion_7d' and 'click_rate_7d' measure for customers?"
answer = rag(query)
print(answer)

The difference between the features 'search_to_purchase_conversion_7d' and 'click_rate_7d' lies in what they measure and the logic used in their computation.

- **search_to_purchase_conversion_7d** measures the conversion rate from searches to purchases over the last 7 days (though specific details about its computation logic and description are not provided in the context). 

- **click_rate_7d**, on the other hand, measures the campaign click-through rate over the last 7 days. It is computed as the count of clicks divided by the count of impressions for each campaign, indicating how effective a marketing campaign is in generating engagements from impressions.

In summary, while 'search_to_purchase_conversion_7d' focuses on conversion from search actions to sales, 'click_rate_7d' measures the effectiveness of campaign clicks relative to impressions.


## Retrieval evaluation

In [15]:
# Ground truth generation for retrieval evaluation
df_question = pd.read_csv('../data/ground-truth-retrieval.csv')

In [16]:
df_question.head()

,id,question
0,0,What does the purchase_count_7d feature repres...
1,0,How is the purchase_count_7d feature calculate...
2,0,What are the update frequency and serving stor...
3,0,Which models utilize the purchase_count_7d fea...
4,0,What specific data source is used to compute t...


In [17]:
ground_truth = df_question.to_dict(orient='records')

In [18]:
ground_truth[10]

{'id': 2,
 'question': 'What does the purchase_count_30d feature measure regarding customer behavior?'}

In [19]:
def hit_rate(relevance_total):
    cnt = 0

    for line in relevance_total:
        if True in line:
            cnt = cnt + 1

    return cnt / len(relevance_total)

def mrr(relevance_total):
    total_score = 0.0

    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank] == True:
                total_score = total_score + 1 / (rank + 1)

    return total_score / len(relevance_total)

In [20]:
def minsearch_search(query):
    boost = {}

    results = index.search(
        query=query,
        filter_dict={},
        boost_dict=boost,
        num_results=10
    )

    return results

In [21]:
def evaluate(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        doc_id = q['id']
        results = search_function(q)
        relevance = [d['id'] == doc_id for d in results]
        relevance_total.append(relevance)

    return {
        'hit_rate': hit_rate(relevance_total),
        'mrr': mrr(relevance_total),
    }

In [22]:
from tqdm.auto import tqdm

In [23]:
evaluate(ground_truth, lambda q: minsearch_search(q['question']))

  0%|          | 0/1200 [00:00<?, ?it/s]

{'hit_rate': 0.98, 'mrr': 0.6296223544973527}

## Finding the best parameters

In [24]:
df_validation = df_question[:100]
df_test = df_question[100:]

In [25]:
import random

def simple_optimize(param_ranges, objective_function, n_iterations=10):
    best_params = None
    best_score = float('-inf')  # Assuming we're minimizing. Use float('-inf') if maximizing.

    for _ in range(n_iterations):
        # Generate random parameters
        current_params = {}
        for param, (min_val, max_val) in param_ranges.items():
            if isinstance(min_val, int) and isinstance(max_val, int):
                current_params[param] = random.randint(min_val, max_val)
            else:
                current_params[param] = random.uniform(min_val, max_val)
        
        # Evaluate the objective function
        current_score = objective_function(current_params)
        
        # Update best if current is better
        if current_score > best_score:  # Change to > if maximizing
            best_score = current_score
            best_params = current_params
    
    return best_params, best_score

In [26]:
gt_val = df_validation.to_dict(orient='records')

In [27]:
def minsearch_search(query, boost=None):
    if boost is None:
        boost = {}

    results = index.search(
        query=query,
        filter_dict={},
        boost_dict=boost,
        num_results=10
    )

    return results

In [28]:
param_ranges = {
    'feature_name': (0.0, 3.0),              # Most important - users ask by name
    'feature_group': (0.0, 3.0),             # Important - users ask by category
    'feature_description': (0.0, 3.0),       # Important - users search by description
    'computation_logic': (0.0, 2.0),         # Medium - technical users look at this
    'models_using_feature': (0.0, 2.0),      # Medium - users ask about model associations
    'data_source': (0.0, 1.5),               # Lower - less frequently searched
    'serving_store': (0.0, 1.0),             # Lower - less frequently searched
    'update_frequency': (0.0, 1.0),          # Lowest - rarely the primary search term
}

def objective(boost_params):
    def search_function(q):
        return minsearch_search(q['question'], boost_params)
    
    results = evaluate(gt_val, search_function)
    return results['mrr']

In [29]:
simple_optimize(param_ranges, objective, n_iterations=20)

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

...

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

({'feature_name': 1.8446328427681924,
  'feature_group': 0.6489343789556048,
  'feature_description': 2.5154580664293107,
  'computation_logic': 1.4827410459811463,
  'models_using_feature': 0.6099692313814562,
  'data_source': 0.41476785579387243,
  'serving_store': 0.400017718539998,
  'update_frequency': 0.04473157209925138},
 0.5285)

In [31]:
def minsearch_improved(query):
    boost = {
        'feature_name': 1.84,
        'feature_group': 0.65,
        'feature_description': 2.52,
        'computation_logic': 1.48,
        'models_using_feature': 0.61,
        'data_source': 0.41,
        'serving_store': 0.40,
        'update_frequency': 0.04
    }

    results = index.search(
        query=query,
        filter_dict={},
        boost_dict=boost,
        num_results=10
    )

    return results

evaluate(ground_truth, lambda q: minsearch_improved(q['question']))

  0%|          | 0/1200 [00:00<?, ?it/s]

{'hit_rate': 0.98, 'mrr': 0.6296223544973527}

In [33]:
# Step 1: Split your ground truth
df = pd.read_csv('../data/ground-truth-retrieval.csv')
df_val = df.sample(frac=0.8, random_state=42)   # 80% for optimization
df_test = df.drop(df_val.index)                  # 20% for final evaluation

gt_val = df_val.to_dict(orient='records')
gt_test = df_test.to_dict(orient='records')

# Step 2: Run optimization on validation set only
best_params, best_score = simple_optimize(param_ranges, objective, n_iterations=50)

# Step 3: Evaluate on test set
def minsearch_optimized(query):
    results = index.search(
        query=query,
        filter_dict={},
        boost_dict=best_params,
        num_results=10
    )
    return results

# Baseline (no boost) on test set
baseline_results = evaluate(gt_test, lambda q: minsearch_search(q['question']))
print("Baseline on test:", baseline_results)

# Optimized on test set
optimized_results = evaluate(gt_test, lambda q: minsearch_optimized(q['question']))
print("Optimized on test:", optimized_results)

  0%|          | 0/960 [00:00<?, ?it/s]

  0%|          | 0/960 [00:00<?, ?it/s]

  0%|          | 0/960 [00:00<?, ?it/s]

  0%|          | 0/960 [00:00<?, ?it/s]

  0%|          | 0/960 [00:00<?, ?it/s]

...

  0%|          | 0/960 [00:00<?, ?it/s]

  0%|          | 0/240 [00:00<?, ?it/s]

Baseline on test: {'hit_rate': 0.9916666666666667, 'mrr': 0.6368650793650797}


  0%|          | 0/240 [00:00<?, ?it/s]

Optimized on test: {'hit_rate': 0.9916666666666667, 'mrr': 0.6368650793650797}
